In  a docker terminal run
jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root

then open jupyter in the printed link (such as http://127.0.0.1:8888/tree?token=4d9d578397aede50fc5bd2d92794b562a6bbcbc73b2dfe51)

CODE HERE, REFRESH AND EXECUTE IN THE BROWSER

In [1]:
import os
import sys
import django

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

sys.path.append('/app')

os.environ['DJANGO_SETTINGS_MODULE'] = 'app.settings'
os.environ['PYTHONPATH'] = '/app'

django.setup()


In [2]:
from django.conf import settings
import requests
from requests.auth import HTTPBasicAuth
import json
import rasterio
import numpy as np
from toolbox import models
from geo.Geoserver import Geoserver

In [14]:
geo = Geoserver(settings.GEOSERVER_URL, username=settings.GEOSERVER_USER, password=settings.GEOSERVER_PASS)
path='/app/raster_data/mar_result_9.tif'
workspace = 'spreewassern_raster'
style_name="style_raster_percent_sieker_2"
layer_name = 'mar_result_9'

In [4]:
geo.create_workspace(workspace='spreewassern_vector')

'201 Workspace spreewassern_vector created!'

In [13]:
os.environ["DB_PASS"]

'postgis'

In [23]:
geo.create_featurestore(
        store_name='above_ground_catchment_area',
        workspace='spreewassern_vector',
        db=os.environ["DB_NAME"],
        host=os.environ["DB_HOST"],  # <-- "db"
        port='5432',
        pg_user=os.environ["DB_USER"],
        pg_password=os.environ["DB_PASS"],
        schema='public'
    )


'Featurestore created/updated successfully'

In [25]:
geo.publish_featurestore(workspace='spreewassern_vector', store_name='above_ground_catchment_area', pg_table='toolbox_abovegroundcatchmentarea')

201

In [17]:
geo.publish_featurestore(workspace='spreewassern_vector', store_name='teststore_data', pg_table='toolbox_ezg25')

201

In [8]:
settings.DATABASES[

{'default': {'ENGINE': 'django.contrib.gis.db.backends.postgis',
  'HOST': 'db',
  'NAME': 'postgis',
  'USER': 'postgis',
  'PASSWORD': 'postgis',
  'ATOMIC_REQUESTS': False,
  'AUTOCOMMIT': True,
  'CONN_MAX_AGE': 0,
  'CONN_HEALTH_CHECKS': False,
  'OPTIONS': {},
  'TIME_ZONE': None,
  'PORT': '',
  'TEST': {'CHARSET': None,
   'COLLATION': None,
   'MIGRATE': True,
   'MIRROR': None,
   'NAME': None}}}

In [ ]:
geo.create_featurestore(store_name='ezg25', workspace='spreewassern_vector', db='postgris', host='localhost', pg_user='postgres',
                        pg_password='admin')




In [61]:
geo.create_coveragestore(layer_name=layer_name, path=path, workspace='api_demo')

{'coverageStore': {'name': '9_mar_result',
  'type': 'GeoTIFF',
  'enabled': True,
  'workspace': {'name': 'api_demo'},
  '_default': False,
  'dateCreated': '2025-11-20 12:37:56.894 UTC',
  'disableOnConnFailure': False,
  'url': 'file:data/api_demo/9_mar_result/9_mar_result.geotiff'}}

In [64]:
geo.publish_style(layer_name=layer_name, style_name=style_name, workspace='api_demo')


200

In [43]:
base_url = f"{settings.GEOSERVER_URL}/rest/workspaces/{workspace}/"
auth = HTTPBasicAuth(settings.GEOSERVER_USER, settings.GEOSERVER_PASS)
auth

In [51]:
delete_url = f"{base_url}/coveragestores/{layer_name}.json"
resp=requests.delete(del_url, auth=auth)

In [52]:
resp

<Response [404]>

In [38]:
del_url = f"{settings.GEOSERVER_URL}/rest/workspaces/{workspace}/coveragestores/{layer_name}?recurse=true"
print(del_url)
resp=requests.delete(del_url, auth=auth)


http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/9_mar_result?recurse=true


In [44]:
workspaces_json = f"{base_url}coveragestores.json"
r = requests.get(workspaces_json, auth=auth)
print(r.status_code,)
print( r.text)

200
{"coverageStores":{"coverageStore":[{"name":122,"href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/122.json"},{"name":"Entwaesserungswahrscheinlichkeit_9Parameter_v2","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/Entwaesserungswahrscheinlichkeit_9Parameter_v2.json"},{"name":"a1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/a1.json"},{"name":"a2","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/a2.json"},{"name":"aquifer_classified_v1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/aquifer_classified_v1.json"},{"name":"depth_to_gw_classified_v1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/depth_to_gw_classified_v1.json"},{"name":"dgm200_single_v1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/c

In [24]:
resp.headers

{'X-Frame-Options': 'SAMEORIGIN', 'X-Content-Type-Options': 'nosniff', 'Content-Encoding': 'gzip', 'Content-Type': 'text/plain', 'Transfer-Encoding': 'chunked', 'Date': 'Thu, 20 Nov 2025 07:48:45 GMT', 'Keep-Alive': 'timeout=20', 'Connection': 'keep-alive'}

In [25]:
url = f"{settings.GEOSERVER_URL}/rest/workspaces/spreewassern_raster/coveragestores.json"
resp = requests.get(url, auth=HTTPBasicAuth(settings.GEOSERVER_USER, settings.GEOSERVER_PASS))
print(resp.status_code)
print(resp.text)

200
{"coverageStores":{"coverageStore":[{"name":122,"href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/122.json"},{"name":"14_mar_result","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/14_mar_result.json"},{"name":"Entwaesserungswahrscheinlichkeit_9Parameter_v2","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/Entwaesserungswahrscheinlichkeit_9Parameter_v2.json"},{"name":"a1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/a1.json"},{"name":"a2","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/a2.json"},{"name":"aquifer_classified_v1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/aquifer_classified_v1.json"},{"name":"depth_to_gw_classified_v1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/d

In [26]:
url = f"{settings.GEOSERVER_URL}/rest/about/version.json"
resp = requests.get(url, auth=HTTPBasicAuth(settings.GEOSERVER_USER, settings.GEOSERVER_PASS))
print(resp.status_code, resp.text)

200 {"about":{"resource":[{"@name":"GeoServer","Build-Timestamp":"15-Jun-2024 08:44","Version":"2.25.2","Git-Revision":"06f15aca7f2eea2004299f460b944d59b83552bc"},{"@name":"GeoTools","Build-Timestamp":"15-Jun-2024 07:28","Version":31.2,"Git-Revision":"c77a759972c0d759d5bd28675a614ada2e3fb774"},{"@name":"GeoWebCache","Version":"1.25.2","Git-Revision":"1.25.x/160ecc7173032aaed73b77fdcd4e64f8e884b256"}]}}


In [27]:
with open(f"/app/raster_data/{layer_name}.tif", "rb") as f:
        r = requests.post(
            url,
            auth=HTTPBasicAuth(settings.GEOSERVER_USER, settings.GEOSERVER_PASS),
            headers={"Content-type": "image/tiff"},
            params={"configure": "all", "coverageName": layer_name},
            data=f
        )

In [31]:
r.status_code
r.text

''

In [35]:
layer_url = f"{settings.GEOSERVER_URL}/rest/layers/{workspace}:{layer_name}"
    style_xml = f"""
    <layer>
        <defaultStyle>
            <name>{style_name}</name>
        </defaultStyle>
    </layer>
    """
    r = requests.post(
        layer_url,
        auth=HTTPBasicAuth(settings.GEOSERVER_USER, settings.GEOSERVER_PASS),
        headers={"Content-type": "application/xml"},
        data=style_xml
    )

IndentationError: unexpected indent (4100000396.py, line 2)

In [36]:
resp = requests.get(
    "http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores.json",
    auth=HTTPBasicAuth(settings.GEOSERVER_USER, settings.GEOSERVER_PASS)
)

In [37]:
resp.text

'{"coverageStores":{"coverageStore":[{"name":122,"href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/122.json"},{"name":"Entwaesserungswahrscheinlichkeit_9Parameter_v2","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/Entwaesserungswahrscheinlichkeit_9Parameter_v2.json"},{"name":"a1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/a1.json"},{"name":"a2","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/a2.json"},{"name":"aquifer_classified_v1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/aquifer_classified_v1.json"},{"name":"depth_to_gw_classified_v1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/coveragestores/depth_to_gw_classified_v1.json"},{"name":"dgm200_single_v1","href":"http://geoserver:8080/geoserver/rest/workspaces/spreewassern_raster/cove

In [101]:
drain_pr = models.ToolboxProject.objects.get(pk=53)
project = drain_pr.project_data

In [152]:
def mar_calculate_area(project):
   
    map_labels = models.MapLabels.objects.all()
    suitability_dict = {}
    for label in map_labels:
        suitability = label.suitability
        name = label.name
        map_value = label.map_value
        default_score = label.default_score
        if suitability not in suitability_dict:
            suitability_dict[suitability] = {'mapping': {}}
        suitability_dict[suitability]['map_path'] = label.map_name
        suitability_dict[suitability]['weight'] = int(project.get(f'weighting_{suitability}', 5))/5
        suitability_dict[suitability]['mapping'][name] = {
            'map_value': map_value,
            'default_score': default_score/5,
            'score': int(project.get(f'{suitability}_{name}', default_score))/5,
            }




    return suitability_dict

In [153]:
suitability_dict = mar_calculate_area(project)
suitability_dict

{'hydraulic_conductivity': {'mapping': {'conductivity_lt_5': {'map_value': 10,
    'default_score': 0.2,
    'score': 0.2},
   'conductivity_20_to_30': {'map_value': 200,
    'default_score': 0.8,
    'score': 0.8},
   'conductivity_gt_30': {'map_value': 300,
    'default_score': 1.0,
    'score': 1.0},
   'conductivity_5_to_10': {'map_value': 50,
    'default_score': 0.4,
    'score': 0.4},
   'conductivity_10_to_20': {'map_value': 100,
    'default_score': 0.6,
    'score': 0.6}},
  'map_path': 'hydraulic_conductivity_classified_v1.tif',
  'weight': 0.6},
 'land_use': {'mapping': {'forest_closed_coniferous': {'map_value': 111,
    'default_score': 1.0,
    'score': 1.0},
   'urban': {'map_value': 50, 'default_score': 0.0, 'score': 0.0},
   'permanent_waterbodies': {'map_value': 80,
    'default_score': 0.0,
    'score': 0.0},
   'herbaceous_wetland': {'map_value': 90, 'default_score': 0.0, 'score': 0.0},
   'forest_closed_deciduous': {'map_value': 114,
    'default_score': 1.0,
    '

In [154]:
for key in suitability_dict:
    print(suitability_dict[key]['map_path'])
    #suitability_dict[key]['map_path'] = suitability_dict[key]['map_path'].split('/')[1]
#suitability_dict

hydraulic_conductivity_classified_v1.tif
land_use.tif
aquifer_classified_v1.tif
distance_to_extraction_wells_v1.tif
distance_to_source_water_v1.tif
depth_to_gw_classified_v1.tif


In [108]:
def compute_suitability_from_tifs(suitability_dict):
    user = 111
    FLOAT32_NODATA = np.float32(-3.4028235e+38)

    with rasterio.open('no_injection_area_mask_v1.tif') as mask:
        nogo_mask = mask.read(1)
        dst_crs = mask.crs
        dst_transform = mask.transform
        dst_width = mask.width
        dst_height = mask.height
        dst_profile = mask.profile.copy()

    #dst_profile['nodata'] = FLOAT32_NODATA

    length_stack = len(suitability_dict) + 1
    stack = np.zeros((length_stack, dst_height, dst_width), dtype=np.float32)
    weighted_stack = np.zeros((2, dst_height, dst_width), dtype=np.float32)

    stack[0] = nogo_mask
    weighted_stack[0] = nogo_mask


    mask_arr = None  # to store mask for polygon later
    layer_weight_sum = 0
    for key in suitability_dict:
        layer_weight_sum += suitability_dict[key]['weight']
        
    i = 1
    for key in suitability_dict:
       
        path = suitability_dict[key]['map_path']
        
        # try:
        with rasterio.open(path) as src:
            dst_arr = src.read(1).astype(np.float32)
            dst_nodata = src.nodata
            
        new_arr = np.where(dst_arr == dst_nodata, np.nan, dst_arr).astype(np.float32)

        for k in suitability_dict[key]['mapping']:
            new_arr = np.where(
                new_arr==float(suitability_dict[key]['mapping'][k]['map_value']),
                suitability_dict[key]['mapping'][k]['score'],
                new_arr
                )
        stack[i] = new_arr
        weighted_stack[1] = weighted_stack[1] + (new_arr * suitability_dict[key]['weight'] / layer_weight_sum)
        
        i +=1
        # except:
        #     print(path)
    result_2d = np.prod(weighted_stack, axis=0) * 100
    result_2d = np.where(np.isnan(result_2d), FLOAT32_NODATA, result_2d).astype(np.float32)
    dst_profile["nodata"] = FLOAT32_NODATA

    with rasterio.open(f'{user}_mar_result.tif', 'w', **dst_profile) as f:

        f.write(result_2d.astype(np.float32),1)

    i = 0
    for key in suitability_dict:
        i += 1
        print(i)
        with rasterio.open(f'{user}_weighted_stack_{key}.tif', 'w', **dst_profile) as f:

            f.write(stack[i].astype(np.float32),1)
    
    #publish_raster_on_geoserver(f"{user}_mar_result")


    return stack, weighted_stack, result_2d

In [109]:
ret = compute_suitability_from_tifs(suitability_dict)

1
2
3
4
5
6


In [81]:
dst_profile['nodata'] = NODATA

In [173]:
nines = ['9_weighted_stack_aquifer_thickness', '9_weighted_stack_depth_groundwater', '9_weighted_stack_distance_to_source', '9_weighted_stack_distance_to_well', '9_weighted_stack_hydraulic_conductivity', '9_weighted_stack_land_use'] 

In [191]:
#filename= 'no_injection_area_mask'
for filename in ['no_injection_area_mask_v2']:
    with rasterio.open(filename + '.tif') as f:
        vals = f.read(1)
        width = f.width
        height = f.height
        transform = f.transform
        bounds=f.bounds
        nodata=f.nodata
        dst_profile=f.profile.copy()
        crs=f.crs
        print(crs)
        print(dst_profile['dtype'])
        print(dst_profile)
        print('nodata', nodata)
        print('min, max', vals[vals!=nodata].min(), vals[vals!=nodata].max())
    

EPSG:25833
float32
{'driver': 'GTiff', 'dtype': 'float32', 'nodata': -3.4028234663852886e+38, 'width': 732, 'height': 899, 'count': 1, 'crs': CRS.from_wkt('PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25833"]]'), 'transform': Affine(100.0, 0.0, 395183.9815,
       0.0, -100.0, 5839610.3219), 'blockxsize': 732, 'blockysize': 2, 'tiled': False, 'interleave': 'band'}
nodata -3.4028234663852886e+38
min, max 0.0 1.0


In [139]:
nodata = -3.4028235e+38 
dst_profile['nodata'] = nodata

In [134]:
mask

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]], shape=(899, 732))

In [137]:
out = np.where(mask == True, nodata, vals).astype(np.float32)
out

array([[-3.4028235e+38, -3.4028235e+38, -3.4028235e+38, ...,
        -3.4028235e+38, -3.4028235e+38, -3.4028235e+38],
       [-3.4028235e+38, -3.4028235e+38, -3.4028235e+38, ...,
        -3.4028235e+38, -3.4028235e+38, -3.4028235e+38],
       [-3.4028235e+38, -3.4028235e+38, -3.4028235e+38, ...,
        -3.4028235e+38, -3.4028235e+38, -3.4028235e+38],
       ...,
       [-3.4028235e+38, -3.4028235e+38, -3.4028235e+38, ...,
        -3.4028235e+38, -3.4028235e+38, -3.4028235e+38],
       [-3.4028235e+38, -3.4028235e+38, -3.4028235e+38, ...,
        -3.4028235e+38, -3.4028235e+38, -3.4028235e+38],
       [-3.4028235e+38, -3.4028235e+38, -3.4028235e+38, ...,
        -3.4028235e+38, -3.4028235e+38, -3.4028235e+38]],
      shape=(899, 732), dtype=float32)

In [145]:
inputs=[
    'distance_to_extraction_wells_v1', 
        'distance_to_source_water_v1', 
        'depth_to_gw_classified_v1', 
        'aquifer_classified_v1', 
        'hydraulic_conductivity_classified_v1',
    'land_use']

In [144]:
filename= 'no_injection_area_mask_v2'
with rasterio.open(f'{filename}.tif', 'w', **dst_profile) as f:

    f.write(out.astype(np.float32), 1)


In [129]:
mask = vals==nodata
mask

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]], shape=(899, 732))

In [99]:
filename= '111_mar_result'
with rasterio.open(f'{filename}.tif', 'w', **dst_profile) as f:

            f.write(vals.astype(np.float32),1)

In [74]:
NODATA = nodata

In [57]:
dst_profile

{'driver': 'GTiff', 'dtype': 'float32', 'nodata': nan, 'width': 732, 'height': 899, 'count': 1, 'crs': CRS.from_wkt('PROJCS["ETRS89 / UTM zone 33N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25833"]]'), 'transform': Affine(100.0, 0.0, 395183.9815,
       0.0, -100.0, 5839610.3219), 'blockxsize': 732, 'blockysize': 2, 'tiled': False, 'interleave': 'band'}

In [59]:
vals[vals!=nodata]



array([], dtype=float32)

In [16]:
vals[vals==nodata]=dnodata
# vals[vals==vals.min()]=np.nan

In [24]:
vals = np.float32(vals)

In [40]:
vals[vals==nodata]=np.nan
vals[vals==vals.min()]=np.nan

In [41]:
dst_profile['nodata'] = np.nan

In [28]:
new_filename = filename + '_v2.tif'
with rasterio.open(new_filename, 'w', **dst_profile) as f:
    f.write(vals.astype(np.float32),1)
    
    


In [79]:
dem = np.full_like(vals[1], np.nan, dtype=np.float32)
scale_factor = 3000 / 255
dem[vals[3] != 0] = vals[1][vals[3] != 0] * scale_factor

In [80]:
dst_profile['nodata'] = np.nan

In [81]:
dst_profile['count']=1
dst_profile['dtype']='float32'

In [82]:
with rasterio.open('dgm200_single_v1.tif', 'w', **dst_profile) as f:
    f.write(dem.astype(np.float32),1)

In [83]:
filename= 'dgm200_single_v1'
with rasterio.open(filename + '.tif') as f:
    vals = f.read()
    width = f.width
    height = f.height
    transform = f.transform
    bounds=f.bounds
    nodata=f.nodata
    dst_profile=f.profile.copy()
    crs=f.crs
    print(dir(f))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pyx_vtable__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_block_shapes', '_closed', '_count', '_crs', '_crs_wkt', '_descriptions', '_dtypes', '_env', '_gcps', '_get_crs', '_get_rpcs', '_handle_crswkt', '_has_band', '_has_gcps_or_rpcs', '_mask_flags', '_nodatavals', '_offsets', '_read', '_rpcs', '_scales', '_set_all_descriptions', '_set_all_offsets', '_set_all_scales', '_set_all_units', '_set_attrs_from_dataset_handle', '_set_crs', '_set_gcps', '_set_nodatavals', '_set_rpcs', '_transform', '_units', 'block_shapes', 'block_size', 'block_window', 'block_windows', 'bounds', 'checksum', 'close', 'closed', 'colorinterp', 'colormap', 'compression', 'c

In [147]:
np.nan+5

nan

In [150]:
a = np.array([[1, 2, 3], [4, 5, 6]])

In [151]:
a = np.where(a == 2, np.nan, a)
a

array([[ 1., nan,  3.],
       [ 4.,  5.,  6.]])

In [185]:
def compute_suitability_from_tifs(suitability_dict, user):

    with rasterio.open('no_injection_area_mask_v2.tif') as mask:
        nogo_mask = mask.read(1)
        mask_nodata = mask.nodata
        mask_width = mask.width
        mask_height = mask.height
        mask_profile = mask.profile.copy()
    print('1. nogo_mask min, max', nogo_mask[nogo_mask!=mask_nodata].min(), nogo_mask[nogo_mask!=mask_nodata].max())

    nogo_mask = np.where(nogo_mask == mask_nodata, np.nan, nogo_mask)
    print('2. nogo_mask min, max', np.nanmin(nogo_mask), np.nanmax(nogo_mask))

    length_stack = len(suitability_dict)
    stack = np.zeros((length_stack, mask_height, mask_width), dtype=np.float32)
    weighted_stack = np.zeros((mask_height, mask_width), dtype=np.float32)


    layer_weight_sum = 0
    for key in suitability_dict:
        layer_weight_sum += suitability_dict[key]['weight']
        
    i = 0
    for key in suitability_dict:
        
        path = suitability_dict[key]['map_path']
        
        # try:
        with rasterio.open(path) as src:
            dst_arr = src.read(1)  
            dst_nodata = src.nodata
        new_arr = np.where(dst_arr == dst_nodata, np.nan, dst_arr).astype(np.float32)

        for k in suitability_dict[key]['mapping']:
            new_arr = np.where(
                new_arr==float(suitability_dict[key]['mapping'][k]['map_value']),
                suitability_dict[key]['mapping'][k]['score'],
                new_arr
                )
        stack[i] = new_arr
        weighted_stack = weighted_stack + (new_arr * suitability_dict[key]['weight'] / layer_weight_sum)
        print('3. weighted_stack min, max', np.nanmin(weighted_stack), np.nanmax(weighted_stack))
        
        i +=1
        # except:
        #     print(path)
    result_2d = weighted_stack * 100
    print('4. nan min max:', result_2d.min(), result_2d.max(), np.nanmin(result_2d), np.nanmax(result_2d))
    result_2d = np.where(nogo_mask == 0, 0, np.clip(result_2d, 0, 100))
    print('5. nan min max:', result_2d.min(), result_2d.max(), np.nanmin(result_2d), np.nanmax(result_2d))

    with rasterio.open(f'{user}_mar_result.tif', 'w', **mask_profile) as f:

        f.write(result_2d.astype(np.float32),1)

    i = 0
    for key in suitability_dict:
        
        print(i)
        with rasterio.open(f'r{user}_weighted_stack_{key}.tif', 'w', **mask_profile) as f:

            f.write(stack[i].astype(np.float32),1)
        i += 1
    
    #publish_raster_on_geoserver(f"{user}_mar_result")
    print(result_2d)

    # return stack, weighted_stack, result_2d

In [186]:
compute_suitability_from_tifs(suitability_dict, 122)

1. nogo_mask min, max 0.0 1.0
2. nogo_mask min, max 0.0 1.0
3. weighted_stack min, max 0.02857143 0.14285715
3. weighted_stack min, max 0.02857143 0.2857143
3. weighted_stack min, max 0.05714286 0.42857146
3. weighted_stack min, max 0.05714286 0.66666675
3. weighted_stack min, max 0.10476191 0.904762
3. weighted_stack min, max 0.13333334 1.0000001
4. nan min max: nan nan 13.333334 100.000015
5. nan min max: nan nan 0.0 100.0
0
1
2
3
4
5
[[nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]
